# Criar a Base de Dados PostgreSQL

In [11]:
import psycopg2
import getpass
from psycopg2.extensions import ISOLATION_LEVEL_AUTOCOMMIT
from sqlalchemy import create_engine, text
import pandas as pd
import numpy as np
import json
import io
from pathlib import Path

Este notebook automatiza o processo de criação de uma base de dados local no PostgreSQL (`statsbomb_db`) e a ingestão de todas as tabelas processadas a partir dos ficheiros Parquet.

Se definires `LOCAL = True`, a aplicação liga-se à base de dados PostgreSQL local (postgresql://localhost:5432/statsbomb_db); se for `LOCAL = False`, liga-se à base de dados remota utilizando a connection string configurada nos segredos (secrets.toml). No caso deste projeto e para hospedar a base de dados na cloud foi utilizado o Rivestack para deploy da base de dados.

## Estrutura Relacional (Conexões):
- **`competitions`**: Chave Primária composta `(competition_id, season_id)`.
- **`matches`**: Chave Primária `match_id`. Chave Estrangeira ligando a `competitions`.
- **`lineups`**: Chave Primária composta `(match_id, player_id)`. Chave Estrangeira ligando a `matches`.
- **`events`**: Tabela com os eventos explodidos por jogador na frame 360. Chave Estrangeira ligando a `matches`.
- **`line_breaking_passes`**: Métricas de passes que rompem linhas. Chave Estrangeira ligando a `matches`.
- **`reception_ability_index`**: Métricas de índice de receção. Chave Estrangeira ligando a `matches`.

Esta abordagem garante a integridade referencial dos dados e facilita a consulta de métricas agrupadas por jogo ou jogador.

### Inicializar a Base de Dados

Ligamo-nos ao PostgreSQL e criamos a base de dados `statsbomb_db` caso esta ainda não exista. No caso de `LOCAL = False` esta etapa é ignorada.

In [12]:
LOCAL = True

def init_database():
    username = getpass.getuser()
    
    try:
        # Liga-se à base de dados padrão 'postgres' para criar a nova base de dados
        conn = psycopg2.connect(
            host="localhost",
            port=5432,
            user=username,
            database="postgres"
        )
        conn.set_isolation_level(ISOLATION_LEVEL_AUTOCOMMIT)
        cur = conn.cursor()
        
        # Verifica se a base de dados já existe
        cur.execute("SELECT 1 FROM pg_database WHERE datname='statsbomb_db';")
        exists = cur.fetchone()
        
        if not exists:
            cur.execute("CREATE DATABASE statsbomb_db;")
            print("Base de dados criada com sucesso.")
        else:
            print("A base de dados 'statsbomb_db' já existe.")
            
        cur.close()
        conn.close()
    except Exception as e:
        print("Erro ao ligar ou criar a base de dados:", e)

if LOCAL:
    init_database()
else:
    print("Modo remoto: a ignorar criação de base de dados local.")

A base de dados 'statsbomb_db' já existe.


### Criar Tabelas e Relações

Definimos e executamos os comandos SQL DDL para criar as tabelas com os tipos de dados apropriados, chaves primárias, chaves estrangeiras e índices para otimização das consultas.

In [ ]:
import os

if LOCAL:
    db_url = "postgresql://localhost:5432/statsbomb_db"
else:
    # Tenta ler do secrets.toml
    db_url = None
    secrets_path = Path("../.streamlit/secrets.toml")
    if secrets_path.exists():
        with open(secrets_path, "r") as f:
            for line in f:
                if line.strip().startswith("DATABASE_URL"):
                    parts = line.split("=", 1)
                    if len(parts) == 2:
                        db_url = parts[1].strip().strip('"').strip("'")
                        break
                        
    if not db_url:
        db_url = os.environ.get("DATABASE_URL")
        
    if not db_url:
        raise ValueError("DATABASE_URL não encontrada em secrets.toml ou variáveis de ambiente.")
        
    if db_url.startswith("postgres://"):
        db_url = db_url.replace("postgres://", "postgresql://", 1)
        
    print("Ligando à base de dados remota...")

# Engine do SQLAlchemy para a base de dados target
engine = create_engine(db_url)

Esquemas e chaves estrangeiras criados com sucesso no PostgreSQL.


### Helpers de Ingestão Ultra-Rápida

1. **`serialize_complex_cols(df)`**: Serializa arrays multidimensionais (como as posições X e Y do 360) em strings JSON, compatíveis com a coluna JSONB do PostgreSQL.
2. **`copy_df_to_postgres(df, table_name, engine)`**: Grava dados na base de dados usando o comando `COPY` nativo do PostgreSQL através de um buffer em memória.

In [ ]:
def serialize_complex_cols(df):
    df_copy = df.copy()
    for col in df_copy.columns:
        non_nulls = df_copy[col].dropna()
        if not non_nulls.empty:
            first_val = non_nulls.iloc[0]
            if isinstance(first_val, (list, dict, np.ndarray, tuple)):
                print(f"A serializar coluna: {col}")
                df_copy[col] = df_copy[col].apply(
                    lambda x: json.dumps(x.tolist() if isinstance(x, np.ndarray) else x) if x is not None else None
                )
    return df_copy

def copy_df_to_postgres(df, table_name, engine):
    df_processed = serialize_complex_cols(df)
    
    raw_conn = engine.raw_connection()
    try:
        cursor = raw_conn.cursor()
        
        # Obtém as colunas exatas e tipos da tabela para garantir alinhamento e casting correto
        cursor.execute(f"""
            SELECT column_name, data_type 
            FROM information_schema.columns 
            WHERE table_name = '{table_name}' 
            ORDER BY ordinal_position;
        """)
        columns_info = cursor.fetchall()
        db_cols = [row[0] for row in columns_info]
        col_types = {row[0]: row[1] for row in columns_info}
        
        # Remove auto-increment pk do events se aplicável
        if table_name == 'events' and 'event_pk' in db_cols:
            db_cols.remove('event_pk')
            
        # Cria uma cópia das colunas alinhadas
        df_to_copy = df_processed[db_cols].copy()
        
        # Trata tipos numéricos inteiros e booleanos para evitar erros no COPY nativo do Postgres
        for col in df_to_copy.columns:
            db_type = col_types[col].upper()
            if 'INT' in db_type or 'BIGINT' in db_type:
                # Converte floats com NaN (ex: player_id) para Int64 do pandas (sem ponto decimal no CSV)
                df_to_copy[col] = pd.to_numeric(df_to_copy[col], errors='coerce').round().astype('Int64')
            elif 'BOOLEAN' in db_type:
                # Garante que booleanos não são gravados como floats ou valores incompatíveis
                df_to_copy[col] = df_to_copy[col].replace({np.nan: None}).astype(object)
        
        # Cria buffer TSV na memória
        output = io.StringIO()
        df_to_copy.to_csv(output, sep='\t', header=False, index=False, na_rep='\\N')
        output.seek(0)
        
        # Executa comando COPY FROM
        sql_copy = f"COPY {table_name} ({', '.join(db_cols)}) FROM STDIN WITH NULL '\\N' DELIMITER '\t';"
        cursor.copy_expert(sql_copy, output)
        raw_conn.commit()
        print(f"Ingeridos {len(df_to_copy):,} registos com sucesso.")
    except Exception as e:
        raw_conn.rollback()
        print(f"Erro na ingestão de {table_name}:", e)
        raise e
    finally:
        raw_conn.close()

### Ingestão de Todas as Tabelas

Carregamos cada um dos ficheiros Parquet da pasta `data/processed` e inserimos nas tabelas do PostgreSQL na ordem de dependência correta.

In [15]:
DATA_DIR = Path("../data/processed")

# Lista ordenada de tabelas para respeitar chaves estrangeiras
tables = [
    ("competitions", "competitions.parquet"),
    ("matches", "matches.parquet"),
    ("lineups", "lineups.parquet"),
    ("events", "events.parquet"),
    ("line_breaking_passes", "line_breaking_passes.parquet"),
    ("reception_ability_index", "reception_ability_index.parquet")
]

for table_name, file_name in tables:
    file_path = DATA_DIR / file_name
    print(f"\n A processar tabela: '{table_name}' a partir de '{file_name}'")
    
    if not file_path.exists():
        print(f"Ficheiro não encontrado: {file_path}.")
        continue
        
    df = pd.read_parquet(file_path)
    
    # Truncate para garantir idompotência
    with engine.begin() as conn:
        conn.execute(text(f"TRUNCATE TABLE {table_name} CASCADE;"))
        print(f"  Tabela '{table_name}' limpa.")
        
    copy_df_to_postgres(df, table_name, engine)


 A processar tabela: 'competitions' a partir de 'competitions.parquet'
  Tabela 'competitions' limpa.
Ingeridos 1 registos com sucesso.

 A processar tabela: 'matches' a partir de 'matches.parquet'
  Tabela 'matches' limpa.
Ingeridos 51 registos com sucesso.

 A processar tabela: 'lineups' a partir de 'lineups.parquet'
  Tabela 'lineups' limpa.
Ingeridos 1,589 registos com sucesso.

 A processar tabela: 'events' a partir de 'events.parquet'
  Tabela 'events' limpa.
  Serializando coluna complexa: location_x
  Serializando coluna complexa: location_y
  Serializando coluna complexa: pass_end_location
Ingeridos 2,722,393 registos com sucesso.

 A processar tabela: 'line_breaking_passes' a partir de 'line_breaking_passes.parquet'
  Tabela 'line_breaking_passes' limpa.
Ingeridos 1,192 registos com sucesso.

 A processar tabela: 'reception_ability_index' a partir de 'reception_ability_index.parquet'
  Tabela 'reception_ability_index' limpa.
Ingeridos 10,087 registos com sucesso.
